# Model Pipeline

Run Bayesian models across different input scalings and compare DIC.

**Output structure:** `data/models/{csv_stem}/{version}/`

In [1]:
import sys
sys.path.insert(0, '.')

import pandas as pd
from pyjags_pipeline import run_model, compare_dic, list_models

# Available input scalings
DATA_PATHS = {
    'per10k':             'data/wide_weekly_scaledPer10k.csv',
    'per-bed':            'data/wide_weekly_scaledPerBed.csv',
    'per-budget':         'data/wide_weekly_scaledPerBudgetThousands.csv',
    'per-65plus':         'data/wide_weekly_scaledPer1kOver65.csv',
}

print('Available models:', list_models())

Available models: ['v0.1', 'v1.1', 'v2.1', 'v2.2', 'v2.3', 'v2.4', 'v2.5', 'v2.6', 'v2.7', 'v2.8', 'v2.9', 'v3.1', 'v3.2', 'v3.3', 'v3.4', 'v3.5', 'v3.6']


## v4.x — Restructured development ladder (AR added last)

Mean structure is built first, autocorrelation is modelled last. The New Year effect is date-anchored from its introduction.

- v4.1 — Baseline (regional level `alpha[i]` only)
- v4.2 — + annual cycle (52-week cos/sin)
- v4.3 — + regional New Year, date-anchored (`delta_pm1, delta_pre, delta_mid, delta_post`)
- v4.4 — + Mid West reset (5-week block, `sigma_w1..sigma_w5`, gated to MW)
- v4.5 — + AR(1)
- **v4.6 — + AR(2)  [selected base]**

Extensions on the AR(2) base (each independent, all rejected by DIC):

- v4.7 — + 4-week cycle
- v4.8 — + 8-week cycle
- v4.9 — + 26-week cycle

v4.1–v4.4 omit AR, so their iid-Normal likelihood ignores week-to-week autocorrelation; their DIC is used for relative comparison of mean structure only.

In [ ]:
# Per-10k population: full v4.x development ladder (AR added last)
for version in ['v4.1', 'v4.2', 'v4.3', 'v4.4', 'v4.5', 'v4.6', 'v4.7', 'v4.8', 'v4.9']:
    result = run_model(
        version=version,
        data_path=DATA_PATHS['per10k'],
        seed=42,
    )
    print(f'{version}  DIC: {result.dic["DIC"]:.2f}  (deviance {result.dic["deviance"]:.2f}, pD {result.dic["penalty"]:.2f})')

# Reported model (v4.6, AR(2)) refit on the other response scalings used in the thesis
for scale in ['per-bed', 'per-budget', 'per-65plus']:
    result = run_model(
        version='v4.6',
        data_path=DATA_PATHS[scale],
        seed=42,
    )
    print(f'v4.6 [{scale}]  DIC: {result.dic["DIC"]:.2f}')

print('\nDIC comparison (per10k):')
compare_dic(DATA_PATHS['per10k'])[['version', 'description', 'DIC']]